# ErrorInputs — malformed inputs fail cleanly

`MagneticCoupled` · ASN 28 negative (diagnostic) test case (TRUST-LiMe).


## Introduction

Negative (diagnostic) cases of the **coupled low-Rm MHD mode**: each malformed
dataset must be rejected with a non-zero exit code and a readable diagnostic,
rather than crashing silently:

- `error_bad_force` — an invalid keyword inside the `force_lorentz` block is rejected;
- `error_missing_ps` — removing the required `solveur_pression` keyword from the NS equation is rejected;
- `error_two_way_no_ns` — the `vitesse_hydraulique` keyword without a `Navier_Stokes_standard` block is rejected;
- `error_two_way_both_velocities` — declaring both `vitesse_magnetique` and `vitesse_hydraulique` is rejected.

Each case is run **directly** with the `trust` command — not through
`runCases()`, which would abort the notebook on the non-zero exit.

## Contracts checked

| # | Check |
|---|:--:|
| 1 | non-zero exit + readable message |
| 2 | the two-way diagnostics name the keywords / the mutual exclusion |

No mesh-convergence scan for this form: the diagnostic abort is
mesh-independent.


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Run

Run **sequentially** (no parallel launch) so the mesh is built
inline and the malformed keyword is reached.


In [ ]:
import subprocess, tempfile, shutil, os

CASES = [
    "error_bad_force",
    "error_missing_ps",
    "error_two_way_no_ns",
    "error_two_way_both_velocities",
]
OK_MSG = {
    "error_bad_force":  "OK: malformed input rejected with a readable message.",
    "error_missing_ps": "OK: malformed input rejected with a readable message.",
    "error_two_way_no_ns":
        "OK: two-way velocity without Navier-Stokes rejected with a readable message.",
    "error_two_way_both_velocities":
        "OK: ambiguous transport velocity rejected with a readable message.",
}
results = []
for CASE in CASES:
    tmp = tempfile.mkdtemp(prefix=f"lime_{CASE}_")
    shutil.copy(os.path.join("src", f"{CASE}.data"), tmp)
    try:
        p = subprocess.run(["trust", CASE], cwd=tmp,
                           capture_output=True, text=True, timeout=300)
        rc, out = p.returncode, (p.stdout + p.stderr)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

    assert rc != 0, f"{CASE}: expected a non-zero exit, got {rc}"
    assert out.strip(), f"{CASE}: no diagnostic message emitted"
    if CASE == "error_two_way_no_ns":
        assert "vitesse_hydraulique" in out and "Navier_Stokes_standard" in out, \
            f"{CASE}: the diagnostic must name vitesse_hydraulique/Navier_Stokes_standard"
    if CASE == "error_two_way_both_velocities":
        assert "mutually exclusive" in out, \
            f"{CASE}: the diagnostic must report the mutual exclusion"
    tail = [l for l in out.splitlines() if l.strip()][-1:] or [""]
    print(f"{CASE}: rc={rc}  |  {tail[0][:90]}")
    print(OK_MSG[CASE])
    results.append((CASE, rc))


## Result


In [ ]:
# Negative (diagnostic) cases: run directly with `trust`.
print("Negative cases complete: all malformed inputs rejected with a non-zero exit.")
for CASE, rc in results:
    print(f"case: {CASE}  rc={rc}")
